In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
path = '../data/'
df = pd.read_csv(path + 'Telco-churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [4]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = df['TotalCharges'].astype(float)

df.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7032.000000
mean,0.162147,32.371149,64.761692,2283.300441
std,0.368612,24.559481,30.090047,2266.771362
min,0.000000,0.000000,18.250000,18.800000
25%,0.000000,9.000000,35.500000,401.450000
50%,0.000000,29.000000,70.350000,1397.475000
75%,0.000000,55.000000,89.850000,3794.737500
max,1.000000,72.000000,118.750000,8684.800000


## Feature Engineering for Telecom Churn Prediction

Creating domain-specific features relevant to telecom SaaS business:
- Service adoption & engagement metrics
- Customer value & loyalty indicators
- Contract and pricing risk factors
- Customer segment characteristics

In [ ]:
# 1. SERVICE ADOPTION & ENGAGEMENT FEATURES
# Count internet add-on services (security, backup, device protection, tech support)
internet_services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
df['num_internet_addons'] = df[internet_services].apply(
    lambda x: (x == 'Yes').sum(), axis=1
)

# Streaming services count
streaming_services = ['StreamingTV', 'StreamingMovies']
df['num_streaming_services'] = df[streaming_services].apply(
    lambda x: (x == 'Yes').sum(), axis=1
)

# Total add-on services
df['total_services'] = df['num_internet_addons'] + df['num_streaming_services'] + \
                       (df['PhoneService'] == 'Yes').astype(int)

# Has premium support (tech support + online security + device protection)
df['has_premium_support'] = (
    (df['TechSupport'] == 'Yes') & 
    (df['OnlineSecurity'] == 'Yes') & 
    (df['DeviceProtection'] == 'Yes')
).astype(int)

print("Service adoption features created:")
print(f"  - num_internet_addons: {df['num_internet_addons'].value_counts().to_dict()}")
print(f"  - num_streaming_services: {df['num_streaming_services'].value_counts().to_dict()}")
print(f"  - total_services: range {df['total_services'].min()} to {df['total_services'].max()}")

In [ ]:
# 2. CUSTOMER VALUE & LOYALTY FEATURES
# Average monthly charges (handles edge case of new customers)
df['avg_monthly_charges'] = df['TotalCharges'] / (df['tenure'] + 1)

# Customer Lifetime Value (CLV) - normalized
df['clv_ratio'] = df['TotalCharges'] / (df['MonthlyCharges'] + 0.01)

# Monthly charges category (price sensitivity)
df['monthly_charges_tier'] = pd.cut(df['MonthlyCharges'], 
                                     bins=[0, 35, 65, 100, 150], 
                                     labels=['low', 'medium', 'high', 'premium'])

# Tenure groups (customer maturity)
df['tenure_group'] = pd.cut(df['tenure'], 
                            bins=[0, 1, 6, 12, 24, 72], 
                            labels=['very_new', 'new', 'established', 'loyal', 'very_loyal'])

# At-risk early churn (new customers with low tenure)
df['is_early_tenure'] = (df['tenure'] <= 6).astype(int)

print("Customer value features created:")
print(f"  - avg_monthly_charges: ${df['avg_monthly_charges'].mean():.2f}")
print(f"  - clv_ratio: range {df['clv_ratio'].min():.2f} to {df['clv_ratio'].max():.2f}")
print(f"  - tenure_group distribution:\n{df['tenure_group'].value_counts().sort_index()}")

In [ ]:
# 3. CONTRACT & COMMITMENT RISK FEATURES
# Month-to-month is high churn risk
df['is_month_to_month'] = (df['Contract'] == 'Month-to-month').astype(int)

# Contract commitment level (in months)
contract_mapping = {'Month-to-month': 0, 'One year': 12, 'Two year': 24}
df['contract_months'] = df['Contract'].map(contract_mapping)

# Has paperless billing (engagement indicator)
df['has_paperless_billing'] = (df['PaperlessBilling'] == 'Yes').astype(int)

# Automatic payment vs manual (convenience factor)
df['has_automatic_payment'] = df['PaymentMethod'].isin(
    ['Bank transfer (automatic)', 'Credit card (automatic)']
).astype(int)

# Electronic check payment risk (historically associated with higher churn)
df['uses_electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)

print("Contract & commitment features created:")
print(f"  - is_month_to_month: {df['is_month_to_month'].sum()} customers ({df['is_month_to_month'].mean()*100:.1f}%)")
print(f"  - has_automatic_payment: {df['has_automatic_payment'].sum()} customers ({df['has_automatic_payment'].mean()*100:.1f}%)")
print(f"  - uses_electronic_check: {df['uses_electronic_check'].sum()} customers ({df['uses_electronic_check'].mean()*100:.1f}%)")

In [ ]:
# 4. DEMOGRAPHIC & HOUSEHOLD FEATURES
# Customer dependency score (more dependents = stickier)
df['has_dependents'] = (df['Dependents'] == 'Yes').astype(int)
df['has_partner'] = (df['Partner'] == 'Yes').astype(int)
df['household_size'] = df['has_partner'] + df['has_dependents'] + 1  # +1 for self

# Senior citizen flag
df['is_senior'] = df['SeniorCitizen']

# Vulnerable segment (senior without support services)
df['is_vulnerable_senior'] = (
    (df['SeniorCitizen'] == 1) & 
    ((df['TechSupport'] != 'Yes') | (df['OnlineBackup'] != 'Yes'))
).astype(int)

# Young professional segment (high tenure + high services)
df['is_young_professional'] = (
    (df['SeniorCitizen'] == 0) & 
    (df['tenure'] > 12) & 
    (df['total_services'] >= 4)
).astype(int)

print("Demographic & household features created:")
print(f"  - household_size: avg {df['household_size'].mean():.2f} people")
print(f"  - is_senior: {df['is_senior'].sum()} customers")
print(f"  - is_vulnerable_senior: {df['is_vulnerable_senior'].sum()} customers")
print(f"  - is_young_professional: {df['is_young_professional'].sum()} customers")

In [ ]:
# 5. INTERNET SERVICE & INFRASTRUCTURE FEATURES
# Has internet service
df['has_internet'] = (df['InternetService'] != 'No').astype(int)

# Fiber optic adoption (premium internet service)
df['has_fiber_optic'] = (df['InternetService'] == 'Fiber optic').astype(int)

# Internet quality score (internet service + protective services)
df['internet_quality_score'] = (
    (df['InternetService'] != 'No').astype(int) * 2 +
    (df['OnlineBackup'] == 'Yes').astype(int) +
    (df['DeviceProtection'] == 'Yes').astype(int)
)

# Phone service adoption
df['has_phone'] = (df['PhoneService'] == 'Yes').astype(int)

# Multiple communication channels (higher engagement)
df['num_channels'] = (
    df['has_phone'] + 
    df['has_internet'] + 
    (df['MultipleLines'] != 'No phone service').astype(int)
)

print("Internet & infrastructure features created:")
print(f"  - has_internet: {df['has_internet'].sum()} customers ({df['has_internet'].mean()*100:.1f}%)")
print(f"  - has_fiber_optic: {df['has_fiber_optic'].sum()} customers ({df['has_fiber_optic'].mean()*100:.1f}%)")
print(f"  - internet_quality_score: avg {df['internet_quality_score'].mean():.2f}")
print(f"  - num_channels: avg {df['num_channels'].mean():.2f}")

In [ ]:
# 6. CHURN RISK SCORING (Business-oriented composite features)
# Risk factors (higher = more risky)
df['churn_risk_score'] = (
    df['is_month_to_month'] * 3 +           # Month-to-month: highest risk
    df['uses_electronic_check'] * 2 +       # Electronic check payment: risky
    df['is_early_tenure'] * 2 +             # New customers: at-risk period
    (df['total_services'] <= 1).astype(int) * 1.5 +  # Few services: low stickiness
    (1 - df['has_automatic_payment']) * 1   # Manual payment: friction
) / 10  # Normalize to 0-1 scale

# High-value at-risk segment (worth retention focus)
df['high_value_at_risk'] = (
    (df['clv_ratio'] > df['clv_ratio'].quantile(0.75)) &  # High CLV
    (df['churn_risk_score'] > df['churn_risk_score'].quantile(0.75))  # High risk
).astype(int)

# Engagement score (higher = more engaged)
df['engagement_score'] = (
    df['total_services'] / df['total_services'].max() * 0.4 +
    df['has_automatic_payment'] * 0.3 +
    (df['tenure'] / df['tenure'].max()) * 0.3
)

# Customer segment (combining multiple factors)
def assign_segment(row):
    if row['is_vulnerable_senior']:
        return 'Vulnerable Senior'
    elif row['is_young_professional']:
        return 'Young Professional'
    elif row['total_services'] >= 4 and row['contract_months'] > 0:
        return 'Loyal Premium'
    elif row['is_month_to_month'] and row['tenure'] < 6:
        return 'At-Risk New'
    elif row['household_size'] > 1 and row['has_dependents']:
        return 'Family'
    else:
        return 'Standard'

df['customer_segment'] = df.apply(assign_segment, axis=1)

print("Churn risk & engagement features created:")
print(f"  - churn_risk_score: avg {df['churn_risk_score'].mean():.3f}")
print(f"  - high_value_at_risk: {df['high_value_at_risk'].sum()} customers")
print(f"  - engagement_score: avg {df['engagement_score'].mean():.3f}")
print(f"\nCustomer segments distribution:")
print(df['customer_segment'].value_counts())

In [ ]:
# Summary: New features created
engineered_features = [
    # Service engagement
    'num_internet_addons', 'num_streaming_services', 'total_services', 'has_premium_support',
    # Customer value
    'avg_monthly_charges', 'clv_ratio', 'monthly_charges_tier', 'tenure_group', 'is_early_tenure',
    # Contract & commitment
    'is_month_to_month', 'contract_months', 'has_paperless_billing', 'has_automatic_payment', 'uses_electronic_check',
    # Demographics
    'has_dependents', 'has_partner', 'household_size', 'is_senior', 'is_vulnerable_senior', 'is_young_professional',
    # Internet & infrastructure
    'has_internet', 'has_fiber_optic', 'internet_quality_score', 'has_phone', 'num_channels',
    # Risk & engagement
    'churn_risk_score', 'high_value_at_risk', 'engagement_score', 'customer_segment'
]

print("\n" + "="*70)
print("FEATURE ENGINEERING COMPLETE")
print("="*70)
print(f"\nNew features created: {len(engineered_features)}")
print(f"Total columns now: {len(df.columns)}")
print(f"\nOriginal shape: {len(df)} rows, {len(df.columns) - len(engineered_features)} original columns")
print(f"New shape: {len(df)} rows, {len(df.columns)} total columns")

# Display new features
print("\n📊 ENGINEERED FEATURES BY CATEGORY:\n")
print("1️⃣  Service Engagement (4 features):")
print(f"   {engineered_features[0:4]}")
print("\n2️⃣  Customer Value & Loyalty (5 features):")
print(f"   {engineered_features[4:9]}")
print("\n3️⃣  Contract & Commitment (5 features):")
print(f"   {engineered_features[9:14]}")
print("\n4️⃣  Demographics & Household (6 features):")
print(f"   {engineered_features[14:20]}")
print("\n5️⃣  Internet & Infrastructure (5 features):")
print(f"   {engineered_features[20:25]}")
print("\n6️⃣  Risk & Engagement Scores (4 features):")
print(f"   {engineered_features[25:29]}")

print("\n✅ Features ready for modeling!")
print(f"\nDataset info:")
print(df[engineered_features].dtypes)